In [1]:
#| label: setup
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from reranker.embedder import Embedder
from reranker.strategies.hybrid import HybridFusionReranker
from reranker.strategies.distilled import DistilledPairwiseRanker
from reranker.strategies.late_interaction import StaticColBERTReranker
from reranker.strategies.binary_reranker import BinaryQuantizedReranker
from reranker.strategies.pipeline import PipelineReranker, PipelineStage
from reranker.strategies.cascade import CascadeReranker, CascadeConfig, ConfidenceMetric
from reranker.strategies.multi import MultiReranker
from reranker.lexical import BM25Engine

embedder = Embedder()

docs = [
    "In Python, a dataclass field can use default_factory to generate mutable default values like lists or dictionaries.",
    "FastAPI relies on Python dataclasses and Pydantic models for automatic request validation.",
    "The dataclasses.field(default_factory=list) creates a new list for each instance rather than sharing one.",
    "XGBoost models can be serialized to JSON and deployed through ONNX Runtime.",
    "FastAPI async endpoints use asyncio for non-blocking request handling.",
    "Pydantic v2 provides fast validation through a Rust-backed core engine.",
    "ONNX Runtime optimizes ML model inference across hardware platforms.",
    "Python dataclasses support frozen instances and slots optimization since 3.10.",
    "ML deployment pipelines use FastAPI with Docker and Kubernetes orchestration.",
    "The dataclass __post_init__ hook enables custom validation and computed fields.",
    "scikit-learn pipelines combine preprocessing and model steps for reproducible workflows.",
    "Gradient boosted trees achieve state-of-the-art results on tabular data with minimal tuning.",
]

pairs = [
    ("python dataclass default factory", docs[0], 1.0),
    ("python dataclass default factory", docs[2], 0.9),
    ("python dataclass default factory", docs[9], 0.8),
    ("fastapi async validation", docs[1], 1.0),
    ("fastapi async validation", docs[4], 0.9),
    ("ml model deployment", docs[3], 1.0),
    ("ml model deployment", docs[8], 0.9),
    ("ml model deployment", docs[6], 0.7),
]

hybrid = HybridFusionReranker(embedder=embedder)
hybrid.fit_pointwise(
    [p[0] for p in pairs], [p[1] for p in pairs],
    [float(p[2]) for p in pairs], use_regression=True,
)

binary = BinaryQuantizedReranker(embedder=embedder)
binary.fit(
    [p[0] for p in pairs], [p[1] for p in pairs],
    [1 if p[2] >= 0.7 else 0 for p in pairs],
)

colbert = StaticColBERTReranker(embedder=embedder)
colbert.fit(docs)

bm25 = BM25Engine(tokenize_fn=embedder.tokenize)
bm25.fit(docs)

print(f"Hybrid fitted: {hybrid.is_fitted}")
print(f"Binary fitted: {binary.is_fitted}")
print(f"ColBERT fitted: {colbert.is_fitted}")
print(f"BM25 indexed: {len(docs)} docs")

/Users/minghao/Desktop/personal/shallow_cross_encoders/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hybrid fitted: True
Binary fitted: True
ColBERT fitted: True
BM25 indexed: 12 docs


## 2. Pattern 1: PipelineReranker

Progressively filter candidates through stages — each stage reduces the candidate set:

```
Stage 1: BM25          (12 docs → top-8)    ~0.07ms
Stage 2: Binary        (8 docs  → top-5)    ~0.04ms
Stage 3: Hybrid Fusion (5 docs  → top-3)    ~0.45ms
```

In [2]:
#| label: pipeline-demo
pipeline = PipelineReranker()
pipeline.add_stage("bm25_filter", bm25, top_k=8)
pipeline.add_stage("binary_filter", binary, top_k=5)
pipeline.add_stage("hybrid_final", hybrid, top_k=3)

query = "python dataclass default factory"
result = pipeline.run_pipeline(query, docs)

print(f"Query: '{query}'")
print(f"Total latency: {result.total_latency_ms:.2f}ms\n")
print("| Stage | Input | Output | Latency | Top Score |")
print("|-------|-------|--------|---------|-----------|")
for stage in result.stage_results:
    print(f"| {stage['stage_name']:20s} | {stage['input_count']:6d} | {stage['output_count']:6d} | {stage['latency_ms']:.2f}ms | {stage['top_score']:.4f} |")

print(f"\n### Final Top-3")
for r in result.final_ranking:
    print(f"  {r.rank}. [{r.score:.4f}] {r.doc[:80]}...")

Query: 'python dataclass default factory'
Total latency: 1.92ms

| Stage | Input | Output | Latency | Top Score |
|-------|-------|--------|---------|-----------|
| bm25_filter          |     12 |      8 | 0.52ms | 1.0000 |
| binary_filter        |      8 |      5 | 0.31ms | 0.7941 |
| hybrid_final         |      5 |      3 | 1.08ms | 1.1765 |

### Final Top-3
  1. [1.1765] In Python, a dataclass field can use default_factory to generate mutable default...
  2. [0.8099] The dataclasses.field(default_factory=list) creates a new list for each instance...
  3. [0.6578] Python dataclasses support frozen instances and slots optimization since 3.10....


### Pipeline Flow Visualization

In [3]:
#| label: pipeline-viz
fig, ax = plt.subplots(figsize=(10, 4))
stage_names = [s["stage_name"] for s in result.stage_results]
inputs = [s["input_count"] for s in result.stage_results]
outputs = [s["output_count"] for s in result.stage_results]
latencies = [s["latency_ms"] for s in result.stage_results]

x = np.arange(len(stage_names))
w = 0.3
bars_in = ax.bar(x - w/2, inputs, w, label="Input docs", color="#4A90D9", alpha=0.8)
bars_out = ax.bar(x + w/2, outputs, w, label="Output docs", color="#50C878", alpha=0.8)

ax2 = ax.twinx()
ax2.plot(x, latencies, "D-", color="#E8575A", label="Latency", markersize=8, linewidth=2)
ax2.set_ylabel("Latency (ms)", color="#E8575A")

ax.set_xticks(x)
ax.set_xticklabels([s.replace("_", "\n") for s in stage_names], fontsize=9)
ax.set_ylabel("Document Count")
ax.set_title("Pipeline: Progressive Filtering")
ax.set_ylim(0, max(inputs) + 2)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

/var/folders/zv/p_57kc9j1fb9xtj06cw1qb1c0000gn/T/ipykernel_76540/3165362309.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**When to use PipelineReranker:**
- Large candidate sets (>100 docs) where you need to reduce before expensive scoring
- Combining fast lexical filters with expensive semantic models
- Predictable latency budgets per stage

## 3. Pattern 2: CascadeReranker

Use a fast model for most queries, fall back to an expensive one when confidence is low:

In [4]:
#| label: cascade-demo
cascade = CascadeReranker(
    primary=hybrid,
    fallback=binary,
    config=CascadeConfig(
        confidence_threshold=0.3,
        confidence_metric=ConfidenceMetric.TOP_MARGIN,
    ),
)

print("### Cascade: Hybrid (primary) → Binary (fallback)\n")
print(f"{'Query':40s} | {'Confidence':>10s} | {'Fallback':>8s} | {'Latency':>8s}")
print("-" * 80)

test_queries = [
    "python dataclass default factory",
    "xgboost model serialization onnx",
    "fastapi async request validation",
    "completely unrelated query about astronomy",
]

for q in test_queries:
    cascade.reset_stats()
    start = time.perf_counter()
    results = cascade.rerank(q, docs)
    lat = (time.perf_counter() - start) * 1000
    stats = cascade.get_stats()
    conf = results[0].metadata.get("confidence", 0)
    fb = results[0].metadata.get("fallback_used", False)
    print(f"{q:40s} | {conf:10.4f} | {'YES' if fb else 'no':>8s} | {lat:7.2f}ms")

### Cascade: Hybrid (primary) → Binary (fallback)

Query                                    | Confidence | Fallback |  Latency
--------------------------------------------------------------------------------
python dataclass default factory         |     0.3679 |       no |    1.76ms
xgboost model serialization onnx         |     0.1755 |      YES |    1.24ms
fastapi async request validation         |     0.0179 |      YES |    1.20ms
completely unrelated query about astronomy |     0.0028 |      YES |    1.40ms


### Confidence Distribution

In [5]:
#| label: cascade-confidence
from reranker.lexical import BM25Engine as _BM25

confidence_by_metric = {}
for metric in ConfidenceMetric:
    c = CascadeReranker(
        primary=hybrid, fallback=binary,
        config=CascadeConfig(confidence_threshold=0.3, confidence_metric=metric),
    )
    confs = []
    for q in test_queries:
        r = c.rerank(q, docs)
        confs.append(r[0].metadata.get("confidence", 0))
    confidence_by_metric[metric.value] = confs

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(test_queries))
metrics_list = list(confidence_by_metric.keys())
w = 0.2
for i, metric in enumerate(metrics_list):
    ax.bar(x + i * w, confidence_by_metric[metric], w, label=metric, alpha=0.8)
ax.set_xticks(x + w)
ax.set_xticklabels([q[:25] + "..." for q in test_queries], fontsize=7, rotation=30, ha="right")
ax.set_ylabel("Confidence Score")
ax.set_title("Confidence Metrics Comparison")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3, axis="y")
fig.tight_layout()
plt.show()

/var/folders/zv/p_57kc9j1fb9xtj06cw1qb1c0000gn/T/ipykernel_76540/648328936.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**When to use CascadeReranker:**
- You have a fast distilled model + expensive teacher
- You want quality guarantees with speed
- You need observability into fallback rates

## 4. Pattern 3: MultiReranker (Reciprocal Rank Fusion)

Combine multiple independent rankers via RRF — no training required:

In [6]:
#| label: multi-demo
from reranker.strategies.multi import MultiReranker, MultiRerankerConfig

multi = MultiReranker(
    rerankers=[("hybrid", hybrid), ("colbert", colbert), ("bm25", bm25)],
    config=MultiRerankerConfig(weights=[2.0, 1.5, 1.0]),
)

query = "python dataclass default factory"
results = multi.rerank(query, docs)

print(f"Query: '{query}'")
print(f"Strategy: RRF (weights: Hybrid=2.0, ColBERT=1.5, BM25=1.0)\n")
print("| Rank | Score | Document |")
print("|------|-------|----------|")
for r in results[:5]:
    print(f"| {r.rank} | {r.score:.4f} | {r.doc[:70]}... |")

Query: 'python dataclass default factory'
Strategy: RRF (weights: Hybrid=2.0, ColBERT=1.5, BM25=1.0)

| Rank | Score | Document |
|------|-------|----------|
| 1 | 0.0492 | In Python, a dataclass field can use default_factory to generate mutab... |
| 2 | 0.0484 | The dataclasses.field(default_factory=list) creates a new list for eac... |
| 3 | 0.0471 | FastAPI relies on Python dataclasses and Pydantic models for automatic... |
| 4 | 0.0469 | Python dataclasses support frozen instances and slots optimization sin... |
| 5 | 0.0466 | The dataclass __post_init__ hook enables custom validation and compute... |


### RRF Score Breakdown

In [7]:
#| label: rrf-breakdown
individual = {
    "Hybrid": hybrid.rerank(query, docs),
    "ColBERT": colbert.rerank(query, docs),
    "BM25": bm25.rerank(query, docs),
}

doc_ranks = {}
for name, results_ind in individual.items():
    for r in results_ind:
        if r.doc not in doc_ranks:
            doc_ranks[r.doc] = {}
        doc_ranks[r.doc][name] = r.rank

k = 60
weights = {"Hybrid": 2.0, "ColBERT": 1.5, "BM25": 1.0}

print("| Document | Hybrid Rank | ColBERT Rank | BM25 Rank | RRF Score |")
print("|----------|-------------|--------------|-----------|-----------|")
for doc_text, ranks in sorted(doc_ranks.items(), key=lambda x: sum(weights.get(n, 1) / (k + r) for n, r in x[1].items()), reverse=True)[:5]:
    rrf = sum(weights.get(n, 1) / (k + r) for n, r in ranks.items())
    h = ranks.get("Hybrid", "-")
    c = ranks.get("ColBERT", "-")
    b = ranks.get("BM25", "-")
    print(f"| {doc_text[:35]:35s} | {h!s:>11} | {c!s:>12} | {b!s:>9} | {rrf:.4f} |")

| Document | Hybrid Rank | ColBERT Rank | BM25 Rank | RRF Score |
|----------|-------------|--------------|-----------|-----------|
| In Python, a dataclass field can us |           1 |            1 |         1 | 0.0738 |
| The dataclasses.field(default_facto |           2 |            2 |         2 | 0.0726 |
| FastAPI relies on Python dataclasse |           4 |            3 |         4 | 0.0707 |
| Python dataclasses support frozen i |           3 |            4 |         5 | 0.0706 |
| The dataclass __post_init__ hook en |           5 |            5 |         3 | 0.0697 |


### Strategy Agreement Across Patterns

In [8]:
#| label: pattern-comparison
query = "fastapi async validation"
all_docs = docs

pipeline_r = pipeline.rerank(query, all_docs)
cascade_r = cascade.rerank(query, all_docs)
multi_r = multi.rerank(query, all_docs)
hybrid_r = hybrid.rerank(query, all_docs)

patterns = {
    "Hybrid (standalone)": hybrid_r,
    "Pipeline": pipeline_r,
    "Cascade": cascade_r,
    "Multi (RRF)": multi_r,
}

print(f"Query: '{query}'\n")
for name, results in patterns.items():
    top3 = [r.doc[:50] for r in results[:3]]
    print(f"**{name}:** {', '.join(top3)}")

Query: 'fastapi async validation'

**Hybrid (standalone):** FastAPI relies on Python dataclasses and Pydantic , FastAPI async endpoints use asyncio for non-blocki, Pydantic v2 provides fast validation through a Rus
**Pipeline:** FastAPI relies on Python dataclasses and Pydantic , FastAPI async endpoints use asyncio for non-blocki, Pydantic v2 provides fast validation through a Rus
**Cascade:** FastAPI async endpoints use asyncio for non-blocki, FastAPI relies on Python dataclasses and Pydantic , Pydantic v2 provides fast validation through a Rus
**Multi (RRF):** FastAPI relies on Python dataclasses and Pydantic , FastAPI async endpoints use asyncio for non-blocki, Pydantic v2 provides fast validation through a Rus


## 5. Key Takeaways

| Pattern | Use Case | Benefit | Trade-off |
|---------|----------|---------|-----------|
| **Pipeline** | Large candidate sets | Reduces compute progressively | Requires tuning `top_k` per stage |
| **Cascade** | Quality + speed SLA | Fast path 70-90% of time | Fallback latency spikes |
| **Multi (RRF)** | Ensemble diversity | No training, robust | Higher total compute |

**Production recommendation:** Start with `Pipeline → Cascade`:
1. Pipeline filters 1000+ candidates down to 50 using BM25 + Binary
2. Cascade uses Hybrid for 70-90% of queries, falls back to FlashRank when confidence is low
3. Expected: **1-5ms average** vs 40-832ms (FlashRank alone)